# Day 5 — AI Agents & Workflow Automation
## AI Project Failure Predictor

### Hands-on project

We will build the project feature by feature.

**Feature 1:** Collecting project information  
**Feature 2:** Predicting project failure risk  
**Feature 3:** Generating project recommendations

**Final flow:** Project Details → Risk Analysis → Failure Risk Prediction → Recommendations


## Teaching structure

For every feature we will clearly identify:

**Input → Tool/Function → Processing → Output**

This project demonstrates how an AI Agent can analyze project information, identify possible failure risks, and suggest practical corrective actions.


## Colab setup

For faster generation, select:

**Runtime → Change runtime type → T4 GPU**

CPU also works, but the local LLM will be slower.


In [15]:
!pip -q install -U transformers accelerate sentencepiece requests
print("✅ Installation complete")


✅ Installation complete


# 1. Initialize the AI Model

The language model is the reasoning component of the system. It reads project information, analyzes possible risks, and generates recommendations.

We use a small local instruction-following model so the project does not require a paid API key.


In [16]:
import json
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


Device: cpu
Running on CPU


In [17]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_kwargs = {}
if device == "cuda":
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, **model_kwargs
).to(device)

model.eval()
print("✅ Model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [18]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful AI project failure prediction assistant."
):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("✅ LLM helper ready")


✅ LLM helper ready


# 🚀 FEATURE 1 — Collecting Project Information

## Goal

Give the Agent important details of a software project so they can be analyzed for possible failure risks.

### Input
Project name, budget, timeline, team size, team skills, resources, technical complexity, progress, and current issues.

### Output
Structured project information.

This feature demonstrates **Tool Integration**.


In [19]:
def collect_project_info(
    project_name,
    budget,
    timeline,
    team_size,
    team_skills,
    resources,
    technical_complexity,
    progress,
    current_issues
):
    return {
        "project_name": project_name,
        "budget": budget,
        "timeline": timeline,
        "team_size": team_size,
        "team_skills": team_skills,
        "resources": resources,
        "technical_complexity": technical_complexity,
        "progress": progress,
        "current_issues": current_issues
    }

print("✅ Project information tool created")


✅ Project information tool created


In [20]:
project_info = collect_project_info(
    project_name="AI Project Failure Predictor",
    budget="₹50,000",
    timeline="4 months",
    team_size=3,
    team_skills="Python, Machine Learning, basic AI",
    resources="Laptop, dataset, Python libraries, Google Colab",
    technical_complexity="Medium",
    progress="40%",
    current_issues="Limited dataset and limited development time"
)

print("INPUT")
for key, value in project_info.items():
    print(f"{key}: {value}")

print("\nOUTPUT")
print("=" * 60)
print(project_info)


INPUT
project_name: AI Project Failure Predictor
budget: ₹50,000
timeline: 4 months
team_size: 3
team_skills: Python, Machine Learning, basic AI
resources: Laptop, dataset, Python libraries, Google Colab
technical_complexity: Medium
progress: 40%
current_issues: Limited dataset and limited development time

OUTPUT
{'project_name': 'AI Project Failure Predictor', 'budget': '₹50,000', 'timeline': '4 months', 'team_size': 3, 'team_skills': 'Python, Machine Learning, basic AI', 'resources': 'Laptop, dataset, Python libraries, Google Colab', 'technical_complexity': 'Medium', 'progress': '40%', 'current_issues': 'Limited dataset and limited development time'}


## Feature 1 — Understand Input and Output

```text
PROJECT DETAILS
      ↓
Project Information Tool
      ↓
STRUCTURED PROJECT INFORMATION
```

The structured information becomes the input for failure-risk analysis.

**Feature 1 complete ✅**


# 🚀 FEATURE 2 — Project Failure Risk Prediction

The Agent reuses the structured project information from Feature 1 and analyzes factors that could cause project failure.

### Input
Project information

### Output
Risk analysis + failure risk level

```text
Project Information
        ↓
Risk Analysis Tool
        ↓
Failure Risk Prediction
```


In [21]:
def analyze_project_risk(project_info):
    prompt = f"""
Analyze the following software project for possible project failure.

Project Information:
Project Name: {project_info["project_name"]}
Budget: {project_info["budget"]}
Timeline: {project_info["timeline"]}
Team Size: {project_info["team_size"]}
Team Skills: {project_info["team_skills"]}
Resources: {project_info["resources"]}
Technical Complexity: {project_info["technical_complexity"]}
Progress: {project_info["progress"]}
Current Issues: {project_info["current_issues"]}

Tasks:
1. Identify the main factors that may cause project failure.
2. Explain briefly why each factor is risky.
3. Consider budget, timeline, team skills, resources, technical complexity, progress and current issues.
4. Give an overall failure risk level: Low, Medium, or High.
5. Give a short reason for the overall risk.

Use only the information provided. Do not invent facts.
Keep the answer concise.
"""
    return ask_llm(prompt)

print("✅ Project failure risk prediction tool created")


✅ Project failure risk prediction tool created


In [23]:
risk_report = analyze_project_risk(project_info)

print("INPUT")
print("Project:", project_info["project_name"])
print("\nProject information:")
print(project_info)

print("\nOUTPUT")
print("=" * 60)
print(risk_report)


INPUT
Project: AI Project Failure Predictor

Project information:
{'project_name': 'AI Project Failure Predictor', 'budget': '₹50,000', 'timeline': '4 months', 'team_size': 3, 'team_skills': 'Python, Machine Learning, basic AI', 'resources': 'Laptop, dataset, Python libraries, Google Colab', 'technical_complexity': 'Medium', 'progress': '40%', 'current_issues': 'Limited dataset and limited development time'}

OUTPUT
### Analysis of Possible Project Failures:

#### Main Factors That May Cause Project Failure:
1. **Limited Dataset**: The lack of sufficient data can lead to poor model performance and inaccurate predictions.
2. **Insufficient Development Time**: Short deadlines might result in rushed development, leading to bugs and incomplete features.
3. **Inadequate Team Skills**: A small team with limited expertise in machine learning and AI could struggle to develop complex models effectively.
4. **Resource Constraints**: Insufficient hardware (e.g., laptop) and software tools (e.g., 

## Feature 2 — Understand Input and Output

```text
Feature 1
Project Information
      ↓
Feature 2
Risk Prediction Tool
      ↓
Failure Risk Analysis
      ↓
Low / Medium / High
```

This demonstrates **workflow orchestration** because the output of Feature 1 becomes the input to Feature 2.

**Feature 2 complete ✅**


# 🚀 FEATURE 3 — Generating Project Recommendations

The third capability uses the project information and risk analysis to generate practical actions that can reduce the possibility of project failure.

### Input
Project information + Risk analysis

### Output
Risk-reduction recommendations


In [ ]:
def generate_recommendations(project_info, risk_report):
    prompt = f"""
Generate practical recommendations to reduce the risk of project failure.

Project Information:
{project_info}

Risk Analysis:
{risk_report}

Include:
1. Main risk areas
2. Corrective actions
3. Priority actions
4. One short monitoring suggestion

Use only the information provided.
Do not invent project facts.
Keep the recommendations concise and practical.
"""
    return ask_llm(prompt)

print("✅ Project recommendation tool created")


In [ ]:
recommendations = generate_recommendations(project_info, risk_report)

print("INPUT")
print("Project:", project_info["project_name"])
print("\nRisk analysis:")
print(risk_report)

print("\nOUTPUT")
print("=" * 60)
print(recommendations)


## Feature 3 — Understand Input and Output

```text
Project Information + Risk Analysis
                ↓
     Recommendation Tool
                ↓
     Risk-Reduction Actions
```

**Feature 3 complete ✅**


# 🔗 Connect the Three Project Analysis Features

```text
USER
  ↓
PROJECT DETAILS
  ↓
1. COLLECT PROJECT INFORMATION
  ↓
PROJECT INFORMATION
  ↓
2. PREDICT PROJECT FAILURE RISK
  ↓
FAILURE RISK ANALYSIS
  ↓
3. GENERATE RECOMMENDATIONS
  ↓
RISK-REDUCTION ACTION PLAN
```


In [11]:
def run_project_failure_agent(project_details):
    print("AI PROJECT FAILURE PREDICTOR")
    print("=" * 70)

    print("\n[1] Collecting project information...")
    project_info = collect_project_info(**project_details)
    print("✅ Project information collected")

    print("\n[2] Predicting project failure risk...")
    risk_report = analyze_project_risk(project_info)
    print("✅ Failure risk prediction completed")

    print("\n[3] Generating recommendations...")
    recommendations = generate_recommendations(project_info, risk_report)
    print("✅ Recommendations generated")

    return {
        "project_info": project_info,
        "risk_report": risk_report,
        "recommendations": recommendations
    }


project_details = {
    "project_name": "AI Project Failure Predictor",
    "budget": "₹50,000",
    "timeline": "4 months",
    "team_size": 3,
    "team_skills": "Python, Machine Learning, basic AI",
    "resources": "Laptop, dataset, Python libraries, Google Colab",
    "technical_complexity": "Medium",
    "progress": "40%",
    "current_issues": "Limited dataset and limited development time"
}

project_result = run_project_failure_agent(project_details)


AI PROJECT FAILURE PREDICTOR

[1] Collecting project information...
✅ Project information collected

[2] Predicting project failure risk...
✅ Failure risk prediction completed

[3] Generating recommendations...
✅ Recommendations generated


# 📊 Final AI Project Failure Prediction Report


In [12]:
if project_result:
    print("=" * 70)
    print("FINAL AI PROJECT FAILURE PREDICTION REPORT")
    print("=" * 70)

    print("\n1. PROJECT INFORMATION")
    print("-" * 70)
    print(project_result["project_info"])

    print("\n2. FAILURE RISK PREDICTION")
    print("-" * 70)
    print(project_result["risk_report"])

    print("\n3. RECOMMENDATIONS")
    print("-" * 70)
    print(project_result["recommendations"])


FINAL AI PROJECT FAILURE PREDICTION REPORT

1. PROJECT INFORMATION
----------------------------------------------------------------------
{'project_name': 'AI Project Failure Predictor', 'budget': '₹50,000', 'timeline': '4 months', 'team_size': 3, 'team_skills': 'Python, Machine Learning, basic AI', 'resources': 'Laptop, dataset, Python libraries, Google Colab', 'technical_complexity': 'Medium', 'progress': '40%', 'current_issues': 'Limited dataset and limited development time'}

2. FAILURE RISK PREDICTION
----------------------------------------------------------------------
### Analysis of Possible Project Failures:

#### Main Factors That May Cause Project Failure:
1. **Limited Dataset**: The lack of sufficient data can lead to poor model performance and inaccurate predictions.
2. **Insufficient Development Time**: Short deadlines might result in rushed development, leading to bugs and incomplete features.
3. **Inadequate Team Skills**: A small team with limited expertise in machine

# 📚 Connect the Project to the Day 5 Syllabus

| Topic | Demonstration in our project |
|---|---|
| **AI Agents** | AI Project Failure Predictor Agent |
| **Tool Integration** | Project information, risk prediction, and recommendation tools |
| **Planning** | Organizing project analysis steps |
| **Workflow Orchestration** | Project Details → Risk Prediction → Recommendations |
| **Memory** | Store project information and previous analysis |
| **Human-in-the-loop** | Human reviews predicted risk and recommendations |
| **Agent Observability** | Track project analysis steps and status |
| **Multi-Agent Systems** | Project Analysis, Risk Prediction, and Recommendation Agents |


# 8. Memory — Simple Demonstration

Memory allows the Agent system to retain useful project information between interactions.

For this classroom project, we use a simple Python dictionary.


In [13]:
agent_memory = {
    "project_information": project_result["project_info"],
    "previous_risk_analysis": project_result["risk_report"],
    "previous_recommendations": project_result["recommendations"]
}

print("PROJECT MEMORY")
print("=" * 60)
print(json.dumps(agent_memory, indent=2))


PROJECT MEMORY
{
  "project_information": {
    "project_name": "AI Project Failure Predictor",
    "budget": "\u20b950,000",
    "timeline": "4 months",
    "team_size": 3,
    "team_skills": "Python, Machine Learning, basic AI",
    "resources": "Laptop, dataset, Python libraries, Google Colab",
    "technical_complexity": "Medium",
    "progress": "40%",
    "current_issues": "Limited dataset and limited development time"
  },
  "previous_risk_analysis": "### Analysis of Possible Project Failures:\n\n#### Main Factors That May Cause Project Failure:\n1. **Limited Dataset**: The lack of sufficient data can lead to poor model performance and inaccurate predictions.\n2. **Insufficient Development Time**: Short deadlines might result in rushed development, leading to bugs and incomplete features.\n3. **Inadequate Team Skills**: A small team with limited expertise in machine learning and AI could struggle to develop complex models effectively.\n4. **Resource Constraints**: Insufficient h

# 9. Human-in-the-loop

A human reviews the AI-generated failure prediction and recommendations before using them for an important project decision.

This adds a human approval checkpoint.


In [14]:
print("PROJECT FAILURE PREDICTION")
print("=" * 60)
print(project_result["risk_report"])

print("\nRECOMMENDATIONS")
print("-" * 60)
print(project_result["recommendations"])

approval = input(
    "\nApprove this risk prediction and recommendations? (yes/no): "
).strip().lower()

if approval == "yes":
    print("✅ Human approved the prediction and recommendations.")
else:
    print("🛑 Human rejected the prediction and recommendations.")


PROJECT FAILURE PREDICTION
### Analysis of Possible Project Failures:

#### Main Factors That May Cause Project Failure:
1. **Limited Dataset**: The lack of sufficient data can lead to poor model performance and inaccurate predictions.
2. **Insufficient Development Time**: Short deadlines might result in rushed development, leading to bugs and incomplete features.
3. **Inadequate Team Skills**: A small team with limited expertise in machine learning and AI could struggle to develop complex models effectively.
4. **Resource Constraints**: Insufficient hardware (e.g., laptop) and software tools (e.g., Google Colab) can hinder productivity and limit experimentation.
5. **Technical Complexity**: Medium-level technical complexity requires careful planning and execution but also poses challenges due to resource limitations.

#### Explanation Why Each Factor Is Risky:
- **Limited Dataset**: Without enough data, the model will be less accurate, potentially leading to incorrect conclusions abou


KeyboardInterrupt



# 10. Agent Observability

Observability means being able to see what happened during Agent execution.

A simple execution trace records the major steps and their status.


In [ ]:
execution_trace = [
    {"step": 1, "tool": "collect_project_info", "status": "completed"},
    {"step": 2, "tool": "analyze_project_risk", "status": "completed"},
    {"step": 3, "tool": "generate_recommendations", "status": "completed"}
]

print("AGENT EXECUTION TRACE")
print("=" * 60)

for item in execution_trace:
    print(f"Step {item['step']}: {item['tool']} → {item['status']}")


# 11. Multi-Agent Systems

Our working implementation uses **one AI Project Failure Predictor Agent with three capabilities**.

A multi-agent architecture could divide the responsibilities:

```text
                 Manager Agent
                       |
        +--------------+--------------+
        |              |              |
        ▼              ▼              ▼
 Project Analysis   Risk Prediction   Recommendation
     Agent              Agent             Agent
```

Each specialized Agent has one responsibility, while the Manager Agent coordinates the workflow.


# 🎯 Final Project Summary

We built three required capabilities:

### 1. Collecting Project Information
**Input:** Project details  
**Output:** Structured project information

### 2. Predicting Project Failure Risk
**Input:** Project information  
**Output:** Risk analysis and Low/Medium/High failure risk

### 3. Generating Project Recommendations
**Input:** Project information + risk analysis  
**Output:** Risk-reduction recommendations

### Complete architecture

```text
                    USER
                      |
                      ▼
               PROJECT DETAILS
                      |
                      ▼
          ┌─────────────────────────┐
          │ AI PROJECT FAILURE      │
          │     PREDICTOR AGENT     │
          └────────────┬────────────┘
                       |
                       ▼
             ┌─────────────────┐
             │ 1. PROJECT      │
             │ INFORMATION TOOL│
             └────────┬────────┘
                      |
                      ▼
              PROJECT INFORMATION
                      |
                      ▼
             ┌─────────────────┐
             │ 2. RISK         │
             │ PREDICTION TOOL │
             └────────┬────────┘
                      |
                      ▼
             FAILURE RISK LEVEL
                      |
                      ▼
             ┌─────────────────┐
             │ 3. RECOMMENDATION│
             │      TOOL       │
             └────────┬────────┘
                      |
                      ▼
              RISK-REDUCTION
               RECOMMENDATIONS
```

**One project → three features → multiple Agent concepts.**
